In [3]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [4]:
llm_model = "gpt-3.5-turbo"

In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch

file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

embeddings = OpenAIEmbeddings()

db = DocArrayInMemorySearch.from_documents(data, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 5})

llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [6]:
def qa_run(query):
    docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    response = llm.invoke(
        f"""Use the following context to answer the question.

{context}

Question: {query}
"""
    )

    return response.content

In [7]:
print(data[10])
print(data[11])

page_content='Unnamed: 0: 10
name: Cozy Comfort Pullover Set, Stripe
description: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.

Size & Fit
- Pants are Favorite Fit: Sits lower on the waist.
- Relaxed Fit: Our most generous fit sits farthest from the body.

Fabric & Care
- In the softest blend of 63% polyester, 35% rayon and 2% spandex.

Additional Features
- Relaxed fit top with raglan sleeves and rounded hem.
- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.

Imported.' metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10}
page_content='Unnamed: 0: 11
name: Ultra-Lofty 850 Stretch Down Hooded Jacket
description: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of moti

In [8]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

In [9]:
def generate_example(doc):
    prompt = f"""
Generate a question and answer from the following document.

Document:
{doc.page_content}

Return format:
Question: ...
Answer: ...
"""

    response = llm.invoke(prompt).content

    # simple parsing
    lines = response.split("\n")
    question = [l for l in lines if "Question:" in l]
    answer = [l for l in lines if "Answer:" in l]

    return {
        "query": question[0].replace("Question:", "").strip() if question else "",
        "answer": answer[0].replace("Answer:", "").strip() if answer else ""
    }


new_examples = [generate_example(doc) for doc in data[:5]]

print(new_examples[0])
print(data[0])

{'query': "What size should I order for the Women's Campside Oxfords if I wear a half size?", 'answer': "For half sizes not offered, it is recommended to order up to the next whole size for the Women's Campside Oxfords."}
page_content='Unnamed: 0: 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. 

Questions? Please contact us 

In [10]:
examples += new_examples

In [11]:
print(qa_run(examples[0]["query"]))

Yes, the Cozy Comfort Pullover Set does have side pockets.


In [12]:
def qa_debug(query):
    docs = retriever.invoke(query)

    print("\nRetrieved docs:")
    for i, d in enumerate(docs):
        print(f"\n--- Doc {i+1} ---")
        print(d.page_content[:200])

    return qa_run(query)


qa_debug(examples[0]["query"])


Retrieved docs:

--- Doc 1 ---
Unnamed: 0: 73
name: Cozy Cuddles Knit Pullover Set
description: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime 

--- Doc 2 ---
Unnamed: 0: 10
name: Cozy Comfort Pullover Set, Stripe
description: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable 

--- Doc 3 ---
Unnamed: 0: 265
name: Cozy Workout Vest
description: For serious warmth that won't weigh you down, reach for this fleece-lined vest, which provides you with layering options whether you're inside or o

--- Doc 4 ---
Unnamed: 0: 419
name: Cozy Comfort Sweats
description: Our Ultrasoft Sweatpants in a straight-leg style—made from remarkably comfortable french terry. 

Size & Fit
Inseams: Misses’ 30½", Petite 28½", 

--- Doc 5 ---
Unnamed: 0: 151
name: Cozy Quilted Sweatshirt
description: Our sweatshirt is an instant classic with its

'Yes, the Cozy Comfort Pullover Set does have side pockets.'

In [13]:
predictions = []

for ex in examples:
    result = qa_run(ex["query"])
    predictions.append({
        "query": ex["query"],
        "answer": ex["answer"],
        "result": result
    })

In [14]:
def evaluate(example, prediction):
    prompt = f"""
You are grading a QA system.

Question: {example['query']}
Ground Truth Answer: {example['answer']}
Predicted Answer: {prediction['result']}

Is the predicted answer correct?

Respond with one word: CORRECT or INCORRECT
"""

    return llm.invoke(prompt).content.strip()


graded_outputs = [
    evaluate(examples[i], predictions[i])
    for i in range(len(examples))
]

In [15]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set does have side pockets.
Predicted Grade: CORRECT

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Predicted Grade: CORRECT

Example 2:
Question: What size should I order for the Women's Campside Oxfords if I wear a half size?
Real Answer: For half sizes not offered, it is recommended to order up to the next whole size for the Women's Campside Oxfords.
Predicted Answer: If you wear a half size, you should order up to the next whole size for the Women's Campside Oxfords.
Predicted Grade: CORRECT

Example 3:
Question: What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat?
Real Answer: The small size has dimensions of 18" x 28" and the medium

In [16]:
print(graded_outputs[0])

CORRECT
